In [ ]:
   import pandas as pd, numpy as np
   import plotly.express as px, plotly.graph_objects as go
   from plotly.subplots import make_subplots

   WEB = '/content'
   df_integrated      = pd.read_csv(f'{WEB}/integrated_cd_year.csv')
   df_housing_monthly = pd.read_parquet(f'{WEB}/housing_monthly.parquet')
   df_housing_nta     = pd.read_parquet(f'{WEB}/housing_nta.parquet')
   df_centroids       = pd.read_csv(f'{WEB}/cd_centroids.csv')
   df_crashes         = pd.read_parquet(f'{WEB}/crashes.parquet')

### Q5 (Cross-domain) — Where in NYC do housing problems and serious crashes concentrate?
**Why it matters:** If the same districts carry both a heavy housing-maintenance burden and a high number of serious crashes, the city could coordinate housing inspections and street-safety work there instead of treating them separately. The map shows both burdens on one view: bubble size = housing problems, bubble colour = serious crashes (2022–2025 totals per community district).


In [ ]:
def build_q5(df_integrated, df_centroids):
    """Q5: Map of housing problems (bubble size) and serious crashes (colour) per community district."""
    # Sum both measures per district over the years present in the (possibly filtered) data
    d = (df_integrated.groupby('cd_id', as_index=False)
         [['housing_problems', 'crashes']].sum())

    # Attach each district's centre point and readable label for the map
    d = d.merge(df_centroids[['cd_id', 'cd_label', 'borough', 'latitude', 'longitude']],
                on='cd_id', how='inner')

    # Cross-domain ratio for the hover (guard against division by zero)
    d['crashes_per_10k_problems'] = (
        d['crashes'] / d['housing_problems'].replace(0, np.nan) * 10_000
    ).round(1)

    fig = px.scatter_mapbox(
        d,
        lat='latitude', lon='longitude',
        size='housing_problems', color='crashes',
        hover_name='cd_label',
        hover_data={'borough': True,
                    'housing_problems': ':,',
                    'crashes': ':,',
                    'crashes_per_10k_problems': True,
                    'latitude': False, 'longitude': False},
        color_continuous_scale='YlOrRd',
        size_max=35,
        zoom=9.3, center={'lat': 40.71, 'lon': -73.94},

        title='Where housing problems and serious crashes concentrate (by community district)',
        labels={'housing_problems': 'Housing problems',
                'crashes': 'Serious crashes',
                'crashes_per_10k_problems': 'Crashes per 10k problems',
                'borough': 'Borough'},
    )
        fig.update_layout(
        height=600, margin=dict(l=0, r=0, t=50, b=0),
        mapbox_style='white-bg',     # no tile server needed, so it never shows "blocked" errors
        mapbox_layers=[{             # free Esri light-grey street map underneath the bubbles
            'below': 'traces',
            'sourcetype': 'raster',
            'sourceattribution': 'Esri, HERE, Garmin, © OpenStreetMap contributors',
            'source': ['https://server.arcgisonline.com/ArcGIS/rest/services/Canvas/'
                       'World_Light_Gray_Base/MapServer/tile/{z}/{y}/{x}'],
        }],
    )
    return fig

build_q5(df_integrated, df_centroids).show()

In [ ]:
# ============================================================
# Q5: numbers for the answer
# ============================================================
q5 = (df_integrated.groupby(['cd_id', 'cd_label', 'borough'], as_index=False)[['housing_problems', 'crashes']].sum())
q5['crashes_per_10k_problems'] = (q5['crashes'] / q5['housing_problems'] * 10_000).round(1)
print('Top 5 districts by housing problems:')
display(q5.nlargest(5, 'housing_problems')[['cd_label', 'housing_problems', 'crashes']])
print('Top 5 districts by serious crashes:')
display(q5.nlargest(5, 'crashes')[['cd_label', 'housing_problems', 'crashes']])
print('Highest crashes per 10k housing problems (crash-heavy, housing-light):')
display(q5.nlargest(5, 'crashes_per_10k_problems')[['cd_label', 'housing_problems', 'crashes', 'crashes_per_10k_problems']])
print('Share of NYC totals by borough (%):')
b = q5.groupby('borough')[['housing_problems', 'crashes']].sum()
display((b / b.sum() * 100).round(1))

Top 5 districts by housing problems:


,cd_label,housing_problems,crashes
11,Manhattan CD 12,147050,222
15,Bronx CD 4,143687,225
16,Bronx CD 5,123155,179
18,Bronx CD 7,122227,168
9,Manhattan CD 10,111793,160


Top 5 districts by serious crashes:


,cd_label,housing_problems,crashes
53,Queens CD 12,45282,325
48,Queens CD 7,25529,319
41,Brooklyn CD 18,24098,317
24,Brooklyn CD 1,36939,311
28,Brooklyn CD 5,80667,282


Highest crashes per 10k housing problems (crash-heavy, housing-light):


,cd_label,housing_problems,crashes,crashes_per_10k_problems
52,Queens CD 11,5745,149,259.4
4,Manhattan CD 5,9474,223,235.4
57,Staten Island CD 2,7181,145,201.9
58,Staten Island CD 3,6255,124,198.2
0,Manhattan CD 1,4095,73,178.3


Share of NYC totals by borough (%):


,housing_problems,crashes
borough,,
BRONX,34.3,19.9
BROOKLYN,30.0,31.0
MANHATTAN,21.0,20.0
QUEENS,12.7,24.5
STATEN ISLAND,2.0,4.6


**Answer.** The map shows **two different geographies**: housing problems and serious crashes concentrate in
different parts of the city.

- **Housing problems concentrate in the Bronx and upper Manhattan** (the large bubbles in the north). The top 5
  districts are Manhattan CD 12 (147,050 problems), Bronx CD 4 (143,687), Bronx CD 5 (123,155), Bronx CD 7
  (122,227) and Manhattan CD 10 (111,793). Together the Bronx and Manhattan hold **55.3%** of NYC's housing
  problems (34.3% + 21.0%).
- **Serious crashes concentrate in Queens and Brooklyn** (the darkest bubbles in the south and east). The top 5
  districts are Queens CD 12 (325 crashes), Queens CD 7 (319), Brooklyn CD 18 (317), Brooklyn CD 1 (311) and
  Brooklyn CD 5 (282). Four of them have **fewer than 37,000 housing problems**, a quarter of the top housing
  districts.
- **The two burdens overlap only in a few places.** Brooklyn CD 5 is the only district in both top-5 lists.
  Manhattan CD 12 and Bronx CD 4 combine the highest housing totals with above-average crashes (222 and 225),
  matching the double-burdened districts in Q1.
- **Crash-heavy, housing-light districts** have the most serious crashes per 10,000 housing problems: Queens CD 11
  (259.4), Manhattan CD 5 (235.4), Staten Island CD 2 (201.9), Staten Island CD 3 (198.2) and Manhattan CD 1
  (178.3). These have few rental-housing complaints but busy roads.

This is an association only; it does not mean one burden causes the other.

**Benefit to NYC:** because the burdens are geographically separate, housing inspection resources belong mainly
in the Bronx and upper Manhattan, and street-safety resources mainly in Queens and Brooklyn. Joint action fits
only the few overlapping districts.

**Limitation:** each district is drawn as a single centre point, so the map hides variation inside a district.
Raw totals favour large, populous districts (per-capita rates would need population data), and crashes per
10,000 housing problems is very high in districts with few housing problems, partly because the denominator is small.

### Q12 (Transportation) — Are night crashes more severe than day crashes, and does that differ by borough?
**Why it matters:** If serious crashes at night injure or kill more people per crash, safety measures such as lighting, speed enforcement and late-night patrols could be targeted at the hours and boroughs where severity is highest. Night = 20:00–05:59, day = 06:00–19:59. Severity is measured two ways: victims per crash, and the % of crashes with at least one fatality.

In [ ]:
from plotly.subplots import make_subplots

def build_q12(df_crashes):
    """Q12: Day vs night severity of serious crashes per borough (victims/crash and % fatal)."""
    d = df_crashes.dropna(subset=['crash_hour', 'borough']).copy()

    # Night = 20:00-05:59, day = 06:00-19:59
    is_night = (d['crash_hour'] >= 20) | (d['crash_hour'] < 6)
    d['period'] = np.where(is_night, 'Night', 'Day')
    d['has_fatality'] = d['total_fatalities'] > 0

    g = (d.groupby(['borough', 'period'], as_index=False)
           .agg(crashes=('has_fatality', 'size'),
                victims=('total_victims', 'sum'),
                fatal_crashes=('has_fatality', 'sum')))
    g['victims_per_crash'] = (g['victims'] / g['crashes']).round(2)
    g['fatal_pct'] = (g['fatal_crashes'] / g['crashes'] * 100).round(1)

    boroughs = sorted(g['borough'].unique())
    colors = {'Day': '#F2A541', 'Night': '#2E3A87'}

    fig = make_subplots(rows=1, cols=2, subplot_titles=(
        'Victims per crash', '% of crashes with a fatality'))

    for period in ['Day', 'Night']:
        p = g[g['period'] == period].set_index('borough').reindex(boroughs).reset_index()
        hover = ('<b>%{x}</b> – ' + period +
                 '<br>Crashes: %{customdata[0]:,}'
                 '<br>Victims per crash: %{customdata[1]}'
                 '<br>Fatal crashes: %{customdata[2]:,} (%{customdata[3]}%)<extra></extra>')
        custom = p[['crashes', 'victims_per_crash', 'fatal_crashes', 'fatal_pct']].values
        fig.add_trace(go.Bar(x=p['borough'], y=p['victims_per_crash'], name=period,
                             marker_color=colors[period], customdata=custom,
                             hovertemplate=hover, legendgroup=period), row=1, col=1)
        fig.add_trace(go.Bar(x=p['borough'], y=p['fatal_pct'], name=period,
                             marker_color=colors[period], customdata=custom,
                             hovertemplate=hover, legendgroup=period,
                             showlegend=False), row=1, col=2)

    fig.update_layout(barmode='group', height=500,
                      title='Night vs day severity of serious crashes by borough',
                      legend_title_text='Time of day')
    fig.update_xaxes(title_text='Borough')
    fig.update_yaxes(title_text='Victims per crash', row=1, col=1)
    fig.update_yaxes(title_text='Crashes with a fatality (%)', row=1, col=2)
    return fig

build_q12(df_crashes).show()

In [ ]:
# ============================================================
# Q12: numbers for the answer
# ============================================================
d12 = df_crashes.dropna(subset=['crash_hour', 'borough']).copy()
d12['period'] = np.where((d12['crash_hour'] >= 20) | (d12['crash_hour'] < 6), 'Night', 'Day')
d12['fatal'] = d12['total_fatalities'] > 0

overall = d12.groupby('period').agg(crashes=('fatal', 'size'), victims=('total_victims', 'sum'), fatal=('fatal', 'sum'))
overall['share_of_crashes_%'] = (overall['crashes'] / overall['crashes'].sum() * 100).round(1)
overall['victims_per_crash'] = (overall['victims'] / overall['crashes']).round(2)
overall['fatal_%'] = (overall['fatal'] / overall['crashes'] * 100).round(1)
print('Citywide:')
display(overall)

g12 = d12.groupby(['borough', 'period']).agg(crashes=('fatal', 'size'), victims=('total_victims', 'sum'), fatal=('fatal', 'sum'))
g12['victims_per_crash'] = (g12['victims'] / g12['crashes']).round(2)
g12['fatal_%'] = (g12['fatal'] / g12['crashes'] * 100).round(1)
t12 = g12['fatal_%'].unstack('period')
t12['gap_pts'] = (t12['Night'] - t12['Day']).round(1)
print('By borough:')
display(g12[['crashes', 'victims_per_crash', 'fatal_%']].unstack('period'))
display(t12.sort_values('gap_pts', ascending=False))

Citywide:


,crashes,victims,fatal,share_of_crashes_%,victims_per_crash,fatal_%
period,,,,,,
Day,7675,9999,467,64.2,1.30,6.1
Night,4271,6254,453,35.8,1.46,10.6


By borough:


crashes       victims_per_crash       fatal_%      
period            Day Night               Day Night     Day Night
borough                                                          
BRONX            1443   922              1.32  1.48     4.6  12.1
BROOKLYN         2303  1253              1.30  1.43     6.5   9.8
MANHATTAN        1613   832              1.22  1.37     4.9   8.5
QUEENS           1920  1082              1.34  1.54     7.4  11.8
STATEN ISLAND     396   182              1.41  1.60     7.3  10.4

period,Day,Night,gap_pts
borough,,,
BRONX,4.6,12.1,7.5
QUEENS,7.4,11.8,4.4
MANHATTAN,4.9,8.5,3.6
BROOKLYN,6.5,9.8,3.3
STATEN ISLAND,7.3,10.4,3.1


**Answer.** Yes. **Night crashes (20:00–05:59) are more severe than day crashes in every borough**, on both measures.

| | Day | Night |
|---|---|---|
| Share of serious crashes | 64.2% (7,675) | 35.8% (4,271) |
| Victims per crash | 1.30 | **1.46** |
| % of crashes with a fatality | 6.1% | **10.6%** |

Night accounts for only **35.8% of serious crashes but 49.2% of fatal crashes** (453 of 920), and a night crash is
**1.7 times as likely** to involve a death as a day crash.

**By borough**, the night–day fatality gap is largest in the **Bronx** (4.6% by day vs **12.1%** at night, +7.5
points, 2.6 times as likely) and smallest in **Staten Island** (7.3% vs 10.4%, +3.1 points). Queens has the
highest daytime fatality share (7.4%) and the second-highest at night (11.8%). Night crashes also involve more
victims in every borough, from 1.37 per crash in Manhattan to 1.60 in Staten Island.

**Benefit to NYC:** lighting, speed enforcement and late-night patrols would reach the most severe crashes,
especially in the Bronx and Queens. This fits Q10 (Member 2), which found that the hours between 00:00 and
06:00 are the deadliest.

**Limitation:** the data cannot show why night crashes are more severe (speed, visibility, alcohol and traffic
volume are not recorded). Fatal crashes are rare, so borough percentages are noisy, especially Staten Island at
night (182 crashes). The hover shows the crash count behind each bar.